# 1일차 2교시 — MDP 소개

**PyTorch로 배우는 강화학습 · 1일차 Tabular-based Methods · 2026-07-27 (월)**

이애본 (Ph.D Aebon) · DreamIT Biz · https://pytorch26.dreamitbiz.com

---

## 🎯 학습목표

- 마르코프 결정 과정(MDP)의 5요소 (S, A, P, R, γ)를 설명할 수 있다
- 상태가치함수 V(s)와 행동가치함수 Q(s,a)의 차이를 이해한다
- 벨만 방정식의 재귀 구조를 이해한다

---

# ⚡ 실행 방법 두 가지 — 편한 쪽을 고르세요

### 방법 ① 통째로 한 번에
바로 아래 **[통째로 실행]** 셀 **하나만** 실행하면 끝까지 돕니다.
결과부터 보고 싶으신 분께 권합니다.

### 방법 ② 단계별로 하나씩
그 아래 **[단계별]** 부분을 위에서부터 `Shift + Enter` 로 하나씩 실행하세요.
모두 **11칸**입니다. 한 칸 돌리고 결과 보고 넘어가면 됩니다.

> **이 교시는 혼자 돌아갑니다.** 앞 교시를 먼저 실행하지 않아도 됩니다.
> (앞 교시에서 만든 것을 이 노트북 안에 다시 넣어 뒀습니다 — 사이트의 *이 교시 전체 코드* 와 같은 판입니다.)
> 설치할 것도 없습니다 — 코랩에 다 들어 있습니다.

---

# ① 통째로 한 번에 실행

GitHub 에서 원본을 받아 그대로 돌립니다. 원본이 고쳐지면 자동으로 최신을 받습니다.

In [ ]:
!curl -sL https://raw.githubusercontent.com/aebonlee/pytorch26-lab/main/day1/standalone/02_gridworld_mdp.py -o 02_gridworld_mdp.py
!python 02_gridworld_mdp.py

---

# ② 단계별로 하나씩 실행

이 교시 내용이 **11칸**입니다.
위에서부터 `Shift + Enter`.

> ①을 이미 돌리셨어도 상관없습니다. 처음부터 다시 하는 것과 같습니다.

### 1 / 11 칸

In [ ]:
# ============================================================
# 1일차 2교시 — MDP 소개
# 복사해서 그대로 실행하면 됩니다. 고칠 것 없습니다.
# ------------------------------------------------------------
# 이 교시 코드는 앞 교시의 변수·클래스를 이어 씁니다.
# 그래서 이 블록에는 **여기까지 필요한 코드가 전부** 들어 있습니다.
# (수업용 코드만 따로 복사하면 NameError 가 납니다 — 그건 정상입니다.)
# ============================================================

# ── 1교시에서 이어받음 — 강화학습 소개 ──
import numpy as np                          # 숫자 계산 도구 (파이썬의 계산기)

### 2 / 11 칸

In [ ]:
# ============================================================
# 슬롯머신 10대 중에 좋은 걸 찾기 — 탐험과 활용의 첫 만남
# ------------------------------------------------------------
# 슬롯머신이 10대 있습니다. 각각 나오는 돈의 평균이 다릅니다.
# 그런데 어느 게 좋은지는 ==해봐야 알 수 있습니다.==
#
# 여기서 딜레마가 생깁니다.
#   활용(exploit) : 지금까지 제일 좋았던 걸 계속 당긴다
#   탐험(explore) : 다른 것도 가끔 당겨 본다
#
# 활용만 하면? 처음 운 좋게 나온 것에 갇힙니다.
# 탐험만 하면? 좋은 걸 알면서도 계속 딴 걸 당깁니다.
# ============================================================

np.random.seed(0)                           # 결과를 항상 같게 만든다 (수업용)

n_arms = 10                                 # 슬롯머신 10대
true_means = np.random.normal(0, 1, n_arms) # 각 기계의 '진짜' 평균 (우리는 모른다고 치자)

### 3 / 11 칸

In [ ]:
def run_bandit(epsilon, steps=2000):
    """
    epsilon 확률로 아무거나 당기고, 나머지는 제일 좋아 보이는 걸 당긴다.
    steps 번 당겨 보고 평균 수익을 돌려준다.
    """
    Q = np.zeros(n_arms)        # 각 기계가 얼마나 좋은지 내 '추정치'. 0에서 시작.
    N = np.zeros(n_arms)        # 각 기계를 몇 번 당겼는지 세는 통

    rewards = []                # 매번 받은 돈을 기록

    for t in range(steps):      # steps 번 반복
        # ── ① 어느 기계를 당길지 고른다 ──
        if np.random.rand() < epsilon:      # 0~1 사이 무작위 수가 epsilon 보다 작으면
            a = np.random.randint(n_arms)   #   아무거나 고른다 (탐험)
        else:
            a = np.argmax(Q)                #   추정치가 가장 큰 걸 고른다 (활용)

        # ── ② 실제로 당겨 본다 ──
        r = np.random.normal(true_means[a], 1)   # 진짜 평균 근처에서 값이 나온다
                                                 # 1은 흔들림의 크기 (매번 다르게 나온다)

        # ── ③ 결과를 반영해 추정치를 고친다 ──
        N[a] += 1                            # 이 기계를 한 번 더 당겼다고 기록
        Q[a] += (r - Q[a]) / N[a]            # 추정치를 새 결과 쪽으로 조금 옮긴다
        # 이 한 줄이 '평균 구하기'입니다. 다 모아 뒀다 나누지 않고
        # 나올 때마다 조금씩 옮겨 가는 방식입니다. 강화학습 내내 이 모양이 나옵니다.
        #   새 추정 = 옛 추정 + (실제로 나온 것 - 옛 추정) x 얼마나 반영할지

        rewards.append(r)                    # 받은 돈 기록

    return np.mean(rewards)                  # 평균 수익을 돌려준다

### 4 / 11 칸

In [ ]:
# ── epsilon 을 바꿔 가며 비교 ────────────────────────────
print('epsilon = 아무거나 당겨 볼 확률')
print()

### 5 / 11 칸

In [ ]:
for eps in [0.0, 0.01, 0.1, 0.5]:
    print(f"epsilon={eps:4.2f}  평균 보상 = {run_bandit(eps):.3f}")

print("""
결과 읽는 법
  epsilon = 0.00  탐험을 아예 안 합니다.
                  처음 운 좋게 나온 기계에 갇혀서 더 좋은 걸 영영 못 찾습니다.
  epsilon = 0.01  아주 가끔만 둘러봅니다. 조심스럽습니다.
  epsilon = 0.10  보통 쓰는 값. 대개 여기쯤이 가장 좋습니다.
  epsilon = 0.50  절반을 딴 데 씁니다. 좋은 걸 알면서도 낭비합니다.

→ 너무 안 해봐도 안 되고, 너무 많이 해봐도 안 됩니다.
  이 균형이 강화학습 3일 내내 따라다니는 문제입니다.

  1일차 : 사람이 epsilon 을 정해 준다
  2일차 : 처음엔 크게 나중엔 작게 줄여 나간다
  3일차 : 아예 목표에 넣어서 스스로 조절하게 만든다
""")

### 6 / 11 칸

In [ ]:
# ============================================================
# 바꿔 보기
#   1) steps 를 200 으로 줄이면? → 탐험할 시간이 부족해 결과가 나빠집니다
#   2) steps 를 20000 으로 늘리면? → epsilon 이 작아도 결국 좋은 걸 찾습니다
#   3) np.random.seed(0) 의 0을 1, 2 로 바꿔 보세요.
#      순위가 바뀔 수도 있습니다 — 한 번의 결과로 판단하면 안 된다는 뜻입니다.
# ============================================================

# ── 오늘 이 교시 — MDP 소개 ──
import numpy as np                          # 숫자 계산 도구

### 7 / 11 칸

In [ ]:
# ============================================================
# 4x4 격자 세상 만들기 — 오늘 하루 계속 쓸 놀이판
# ------------------------------------------------------------
# 강화학습 문제를 적는 표준 양식이 MDP 입니다. 다섯 가지만 정하면 됩니다.
#   상태(s)  지금 어느 칸에 있나
#   행동(a)  어느 방향으로 갈까
#   보상(r)  그 행동으로 몇 점 받았나
#   전이     그 행동을 하면 어느 칸이 되나
#   감마(γ)  미래를 얼마나 챙길지
#
# 이 파일에서 만드는 놀이판을 3·4교시에서 계속 씁니다.
# ============================================================

N = 4                                       # 4줄 4칸짜리 격자
n_states = N * N                            # 칸이 모두 16개 (0번부터 15번까지)
n_actions = 4                               # 행동 4가지 (상·하·좌·우)

TERMINALS = [0, n_states - 1]               # 0번 칸과 15번 칸에 도착하면 끝
                                            # (왼쪽 위 구석과 오른쪽 아래 구석)

# 칸 번호는 이렇게 매겨져 있습니다:
#    0  1  2  3
#    4  5  6  7
#    8  9 10 11
#   12 13 14 15

### 8 / 11 칸

In [ ]:
def step(s, a):
    """
    s번 칸에서 a 방향으로 가면 어떻게 되는지 알려 주는 함수.
    돌려주는 값: (다음 칸 번호, 받은 점수)

    이 격자는 '결정적'입니다 — 오른쪽으로 가면 반드시 오른쪽으로 갑니다.
    (미끄러지는 얼음판 같은 건 나중에 다룹니다)
    """
    if s in TERMINALS:                      # 이미 끝난 칸이면
        return s, 0                         # 그 자리에 있고 점수도 없다

    r, c = divmod(s, N)                     # 칸 번호를 (몇 번째 줄, 몇 번째 칸)으로 바꾼다
                                            # 예: 5번 칸 → divmod(5,4) = (1, 1) → 1줄 1칸

    if a == 0:   r = max(r - 1, 0)          # 위로  (0번 줄보다 위로는 못 감 → 벽)
    elif a == 1: r = min(r + 1, N - 1)      # 아래로 (마지막 줄보다 아래로는 못 감)
    elif a == 2: c = max(c - 1, 0)          # 왼쪽으로
    elif a == 3: c = min(c + 1, N - 1)      # 오른쪽으로
    # max/min 을 쓰는 이유: 벽에 부딪히면 제자리에 있게 하려고

    return r * N + c, -1                    # (줄, 칸)을 다시 칸 번호로 바꾸고 점수 -1
    # 왜 점수가 -1 일까요?
    #   한 걸음 걸을 때마다 -1점을 받습니다. 즉 ==빨리 끝낼수록 손해가 적습니다.==
    #   "최대한 빨리 목표까지 가라"를 점수로 표현한 것입니다.

### 9 / 11 칸

In [ ]:
# ── 전이표를 미리 만들어 둔다 ────────────────────────────
# P[s][a] 하면 바로 (다음 칸, 점수)가 나옵니다.
# 매번 step() 을 부르는 것보다 빠르고, 3·4교시 코드가 짧아집니다.
P = [[step(s, a) for a in range(n_actions)] for s in range(n_states)]
# 위 한 줄은 이중 반복문입니다. 풀어 쓰면:
#   P = []
#   for s in range(n_states):
#       row = []
#       for a in range(n_actions):
#           row.append(step(s, a))
#       P.append(row)

print('전이표 P 를 만들었습니다.')
print('  P[5][3] =', P[5][3], ' ← 5번 칸에서 오른쪽으로 가면 6번 칸, 점수 -1')
print('  P[0][3] =', P[0][3], ' ← 0번 칸은 끝난 칸이라 그대로, 점수 0')
print()

### 10 / 11 칸

In [ ]:
# ── 아무렇게나 걸어 보기 ─────────────────────────────────
# 정책(policy)이란 "이 칸에서는 이렇게 한다"는 규칙입니다.
# 지금은 아무 규칙 없이 무작위로 걸어 봅니다.

s = 5                                       # 5번 칸에서 출발
trajectory = []                             # 걸어간 기록을 담을 곳

rng = np.random.default_rng(42)             # 무작위 생성기 (42는 결과 고정용 숫자)

### 11 / 11 칸

In [ ]:
while s not in TERMINALS:                   # 끝나는 칸에 닿을 때까지 반복
    a = rng.integers(n_actions)             # 0~3 중 아무거나 (무작위 정책)
    s_next, r = P[s][a]                     # 그 방향으로 가면 어떻게 되나
    trajectory.append((s, a, r))            # 기록: (어디서, 뭘 했고, 몇 점)
    s = s_next                              # 다음 칸으로 이동

print(f"무작위로 걸었더니")
print(f"  걸음 수  : {len(trajectory)}")
print(f"  총 점수  : {sum(t[2] for t in trajectory)}")
print(f"""
  → 한 걸음마다 -1점이니 ==걸음 수와 총 점수는 부호만 다릅니다.==
    아무렇게나 걸으면 한참 헤맵니다.

  다음 시간부터 "어떻게 하면 덜 헤맬까"를 계산으로 찾습니다.
""")

---

## 🔧 바꿔 보기
  1) 출발 칸 s = 5 를 다른 번호로 바꿔 보세요.
     끝나는 칸(0, 15)에 가까울수록 빨리 끝납니다.
  2) rng 의 42 를 1, 2, 3 으로 바꿔 여러 번 돌려 보세요.
     ==같은 무작위 정책인데도 걸음 수가 크게 달라집니다.==
     이 흔들림이 강화학습을 어렵게 만드는 요소 중 하나입니다.
  3) 벽에 부딪히면 제자리에 있는 것을 확인해 보세요.
     print(P[0][0]) 을 찍어 보시면 됩니다 (0번 칸에서 위로 = 제자리).

---

## 막히면

- 사이트의 같은 교시를 보세요 — 실행 결과와 해설이 그대로 있습니다.
  https://pytorch26.dreamitbiz.com/#/day/1/2
- 오류가 나면 **[막힐 때]** 메뉴부터.
  https://pytorch26.dreamitbiz.com/#/help

---

*Ph.D Aebon & Claude Code 협작 전자출판 도서 · © 2026 DreamIT Biz*